In [3]:
import os
import subprocess
import sys

# Ensure subprocess runs with the exact same python binary and environment variables
current_env = os.environ.copy()
python_bin = sys.executable

def run_script(script_name: str, cwd: str = "."):
    """
    Executes a python script inside the current environment and streams/prints the output.
    """
    print(f"\n{'='*20} Running: {script_name} {'='*20}")
    result = subprocess.run(
        [python_bin, script_name],
        cwd=cwd,
        env=current_env,
        capture_output=True,
        text=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR / ERRORS:", result.stderr)
    return result

# 1. Run the Rollout Engine Sanity Check
run_script("rollout_engine.py")

# 2. Run the Full REINFORCE Variance Experiment
run_script("experiment_variance.py")


==================== Running: rollout_engine.py ====================
Target Sequence to find: [3, 7, 5, 2]

--- Rollout Batch (4 Samples) ---
Generated Actions shape: torch.Size([4, 4])
Generated Action IDs:
 tensor([[11,  5, 13,  5],
        [ 9,  7,  8,  9],
        [11,  4, 12, 11],
        [ 2,  4,  9,  5]])
Log Probs shape:         torch.Size([4, 4])
Rewards per sample:      [0.0, 0.25, 0.0, 0.0]


==================== Running: experiment_variance.py ====================
=== Starting Training: use_baseline=False ===
Epoch 030 | Mean Reward: 0.3906 | Grad Norm: 0.8764
Epoch 060 | Mean Reward: 0.7266 | Grad Norm: 1.2555
Epoch 090 | Mean Reward: 0.9141 | Grad Norm: 0.8339
Epoch 120 | Mean Reward: 0.9375 | Grad Norm: 0.6072
Epoch 150 | Mean Reward: 0.9688 | Grad Norm: 0.4470
=== Starting Training: use_baseline=True ===
Epoch 030 | Mean Reward: 0.4922 | Grad Norm: 0.6368
Epoch 060 | Mean Reward: 0.8672 | Grad Norm: 0.4956
Epoch 090 | Mean Reward: 0.9688 | Grad Norm: 0.2594
Epoch 120 |

CompletedProcess(args=['c:\\WHATEVERELSE\\GeekStuff\\conda\\envs\\pygpu\\python.exe', 'experiment_variance.py'], returncode=0, stdout='=== Starting Training: use_baseline=False ===\nEpoch 030 | Mean Reward: 0.3906 | Grad Norm: 0.8764\nEpoch 060 | Mean Reward: 0.7266 | Grad Norm: 1.2555\nEpoch 090 | Mean Reward: 0.9141 | Grad Norm: 0.8339\nEpoch 120 | Mean Reward: 0.9375 | Grad Norm: 0.6072\nEpoch 150 | Mean Reward: 0.9688 | Grad Norm: 0.4470\n=== Starting Training: use_baseline=True ===\nEpoch 030 | Mean Reward: 0.4922 | Grad Norm: 0.6368\nEpoch 060 | Mean Reward: 0.8672 | Grad Norm: 0.4956\nEpoch 090 | Mean Reward: 0.9688 | Grad Norm: 0.2594\nEpoch 120 | Mean Reward: 0.9531 | Grad Norm: 0.1915\nEpoch 150 | Mean Reward: 0.9922 | Grad Norm: 0.0821\n\n--- Empirical Results ---\nGradient Norm Variance (Raw REINFORCE):  0.095453\nGradient Norm Variance (With Baseline):  0.052885\nVariance Reduction Factor:               1.80x\n', stderr='')

## Variance Experiment: Side-by-Side Analysis

The training dynamics differed noticeably across the 150 epochs.

### Summary Table

| Metric / Stage | Raw REINFORCE (use_baseline=False) | REINFORCE with Baseline (use_baseline=True) |
|---|---:|---:|
| Early Learning (Epoch 30) | Mean Reward: 0.3906 | Mean Reward: 0.4922 |
| Mid Training (Epoch 60) | Grad Norm: 1.2555 | Grad Norm: 0.4956 |
| Late Convergence (Epoch 150) | Grad Norm: 0.4470 | Grad Norm: 0.0821 |
| Final Reward | 0.9688 | 0.9922 |

### What is Happening Mathematically

- The baseline agent learns faster out of the gate.
- Raw REINFORCE experiences much larger gradient oscillations during training.
- The baseline agent settles into near-zero gradient noise as it masters the task.
- The baseline agent reaches near-perfect accuracy, approximately $100\%$.

### Interpretation

The comparison shows that adding a baseline makes the policy gradient estimator much more stable and efficient during training.

## Why the Baseline Reduced Variance by $1.80\times$

Here is the final empirical summary:

### Empirical Results

- Gradient Norm Variance (Raw REINFORCE): $0.095453$
- Gradient Norm Variance (With Baseline): $0.052885$
- Variance Reduction Factor: $1.80\times$

### In Raw REINFORCE

With $A = G$:

- All rewards are non-negative, since $G \in [0, 1]$.
- Even mediocre sequences, such as matching one token and getting $G = 0.25$, still push the model to increase those token probabilities because $G > 0$.
- The optimizer moves in mostly positive directions for many batches, which causes high gradient norm volatility with variance $\text{Var} = 0.095$.

### In Baseline-Subtracted REINFORCE

With $A = G - \bar{G}$:

- If the batch average is $\bar{G} = 0.75$, then:
  - a sequence with $G = 1.0$ gets advantage $A = +0.25$ and is boosted,
  - a sequence with $G = 0.5$ gets advantage $A = -0.25$ and is suppressed.
- The baseline centers the updates around zero.
- Mediocre actions are pushed down relative to better actions, which cuts the gradient variance nearly in half, giving a $1.80\times$ reduction.

### Conclusion

This confirms that the foundational policy gradient engine is working as expected.